#**1. IMPORT**


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

#**2. DATA LOADING**

In [6]:
URL = "https://raw.githubusercontent.com/Hokfu/Energy-Consumption-Model/main/Steel_industry_data.csv"
df = pd.read_csv(URL)

#**3.FEATURE ENGINEERING**

In [7]:
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

df['hour'] = df['date'].dt.hour
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# categorical encoding
df['WeekStatus'] = df['WeekStatus'].map({
    'Weekday': 0,
    'Weekend': 1
})

df = pd.get_dummies(df, columns=['Day_of_week'])


#**4.FEATURE SELECTION**

In [8]:
X = df.drop(['Load_Type', 'date', 'NSM'], axis=1)  # NSM ni olib tashladik
y = df['Load_Type']

#**5.TRAIN / TEST SPLIT**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#**6.GRID SEARCH**

In [10]:
params = {
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_


#**7.EVALUATION**

In [11]:
y_pred = best_model.predict(X_test)

print("Train Accuracy:", best_model.score(X_train, y_train))
print("Test Accuracy:", best_model.score(X_test, y_test))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Train Accuracy: 0.9158461757990868
Test Accuracy: 0.9125285388127854

Classification Report:

              precision    recall  f1-score   support

  Light_Load       0.96      0.95      0.95      3615
Maximum_Load       0.88      0.89      0.88      1454
 Medium_Load       0.85      0.87      0.86      1939

    accuracy                           0.91      7008
   macro avg       0.90      0.90      0.90      7008
weighted avg       0.91      0.91      0.91      7008



#**8.CONFUSION MATRIX**

In [12]:
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))



Confusion Matrix:

[[3417   40  158]
 [  35 1288  131]
 [ 112  137 1690]]


#**9.FEATURE IMPORTANCE**

In [13]:
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)
print("\nFeature Importance:\n")
print(feat_imp.sort_values(ascending=False))


Feature Importance:

hour                                    0.613700
Usage_kWh                               0.153269
month                                   0.139127
Leading_Current_Reactive_Power_kVarh    0.040950
Day_of_week_Sunday                      0.026988
Lagging_Current_Power_Factor            0.011352
WeekStatus                              0.005371
Lagging_Current_Reactive.Power_kVarh    0.003515
Leading_Current_Power_Factor            0.002981
day                                     0.002746
CO2(tCO2)                               0.000000
Day_of_week_Monday                      0.000000
Day_of_week_Friday                      0.000000
Day_of_week_Saturday                    0.000000
Day_of_week_Thursday                    0.000000
Day_of_week_Tuesday                     0.000000
Day_of_week_Wednesday                   0.000000
dtype: float64
